In [3]:
import json

with open("./data/chicago_community_areas.geojson", "r") as f:
    data = json.load(f)

print(f"Total features: {len(data['features'])}")
print(f"Properties keys: {list(data['features'][0]['properties'].keys())}")
print(f"Sample feature name: {data['features'][0]['properties']}")

Total features: 77
Properties keys: ['community', 'area', 'shape_area', 'perimeter', 'area_num_1', 'area_numbe', 'comarea_id', 'comarea', 'shape_len']
Sample feature name: {'community': 'DOUGLAS', 'area': '0', 'shape_area': '46004621.1581', 'perimeter': '0', 'area_num_1': '35', 'area_numbe': '35', 'comarea_id': '0', 'comarea': '0', 'shape_len': '31027.0545098'}


In [5]:
import csv
import json
import random

points = []
area_stats = {}

with open("./data/data/cleaned/chicago_crimes_cleaned.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        lat = row['latitude']
        lon = row['longitude']
        crime_type = row['primary_type']
        area = row['community_area']

        # aggregate stats per community area
        if area:
            if area not in area_stats:
                area_stats[area] = {'total': 0, 'types': {}}
            area_stats[area]['total'] += 1
            area_stats[area]['types'][crime_type] = area_stats[area]['types'].get(crime_type, 0) + 1

        # collect points with valid coordinates
        if lat and lon:
            points.append([float(lat), float(lon), crime_type])

# sample 10k points
sampled = random.sample(points, min(10000, len(points)))

# build community stats with top 3 types
community_stats = {}
for area, stats in area_stats.items():
    top3 = sorted(stats['types'].items(), key=lambda x: x[1], reverse=True)[:3]
    community_stats[area] = {
        'total': stats['total'],
        'top3': [[t, c] for t, c in top3]
    }

# save files
with open("./data/sampled_points.json", "w") as f:
    json.dump(sampled, f)

with open("./data/community_stats.json", "w") as f:
    json.dump(community_stats, f)

print(f"Total points with coordinates: {len(points)}")
print(f"Sampled points: {len(sampled)}")
print(f"Community areas with stats: {len(community_stats)}")

Total points with coordinates: 8381516
Sampled points: 10000
Community areas with stats: 78
